# Tutorial 00: Getting started - Simulating the Galactic neutron star population

Here, we simulate a population of isolated neutron stars and model the detection using three different surveys performed with Murriyang, the Parkes Radio Telescope:

1. the Parks Multibeam Pulsar Survey (PMPS; [Manchester et al. 2001](https://ui.adsabs.harvard.edu/abs/2001MNRAS.328...17M/abstract), [Lorimer et al. 2006](https://ui.adsabs.harvard.edu/abs/2006MNRAS.372..777L/abstract)),
2. the Swinburne Parkes Multibeam Pulsar Survey (SMPS; [Edwards et al. 2001](https://ui.adsabs.harvard.edu/abs/2001MNRAS.326..358E/abstract), [Jacoby et al. 2009](https://ui.adsabs.harvard.edu/abs/2009ApJ...699.2009J/abstract)),
3. the mid- and low-latitude High Time Resolution Universe survey (HTRU; [Keith et al. 2010](https://ui.adsabs.harvard.edu/abs/2010MNRAS.409..619K/abstract)).

In [ ]:
import argparse
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pathlib
import sys
from matplotlib.collections import PathCollection
from matplotlib.legend_handler import HandlerPathCollection, HandlerLine2D


def update(handle, orig):
    handle.update_from(orig)
    handle.set_alpha(1)
    handle.set_markersize(3)


import utilities.plot_settings

# Importing the configuration file to change simulation parameters.
from mlpoppyns.simulator.config_simulator import cfg
import mlpoppyns.simulator.stellar_dynamics.coordinate_conversions as cc
from mlpoppyns.simulator.simulate_population_full import simulate_population

WARNING: If you see a warning here, make sure to set the path to the repository in the simulator configuration file. Otherwise, the examples below will not run.

## Setting up and running the simulation

After importing the base simulator configuration above, we will vary the following simulation parameters in this simple example:

1. `NS_number`: the number of neutron stars we simulate.
2. `t_age_max`: the maximum age for the simulated neutron stars in [yr].
3. `B_initial_log10_mean` and `B_initial_log10_sigma`: the mean and standard deviation of the Gaussian distribution for the $\log_{10}$ of the initial magnetic field.
4. `P_initial_log10_mean` and `P_initial_log10_sigma`: the mean and standard deviation of the Gaussian distribution for the $\log_{10}$ of the initial spin period.

Note that the details of all free model parameters are specified in the `mlpoppyns/simulator/config_simulator.json`. If we wanted, we could modify all of these parameters, as we do in the cell below for the number of neutron stars and the initial spin period and magnetic field distributions.

In [ ]:
cfg["NS_number"] = 300000
cfg["t_age_max"] = 3.0e7
cfg["B_initial_log10_mean"] = 13.1
cfg["B_initial_log10_sigma"] = 0.45
cfg["P_initial_log10_mean"] = -1.0
cfg["P_initial_log10_sigma"] = 0.38

We next specify the output directory where the simulation results will be saved.

In [ ]:
output_dir = "output/sim_full"

We now run the simulation by calling the `simulate_population` function from the `mlpoppyns.simulator.simulate_population_full` module with our specific parameter choices.

WARNING: If a `FileNotFound` error appears, remember to check the path to the repository in the `mlpoppyns/simulator/config_simulator.json` configuration file as outlined in the GitHub `README.md` and the `Getting started` page of our documentation.

In [ ]:
simulation_args = argparse.Namespace(
    save_dir=output_dir,
    parameter_override=None,
)
simulate_population(simulation_args)

## Reading the simulation results

We first read in our compressed `.pkl` files:
* The `final_population.pkl.gz` contains the final population properties after the evolution.
* One `.pkl.gz` files for each of the three modeled surveys (containing the stars detected by that survey).

In [ ]:
data_full = pd.read_pickle(
    pathlib.Path().joinpath(output_dir, "final_population.pkl.gz"),
    compression="gzip",
)
data_full.columns

In [ ]:
data_PMPS = pd.read_pickle(
    pathlib.Path().joinpath(output_dir, "survey_PMPS_results.pkl.gz"),
    compression="gzip",
)
data_PMPS.columns

In [ ]:
data_SMPS = pd.read_pickle(
    pathlib.Path().joinpath(output_dir, "survey_SMPS_results.pkl.gz"),
    compression="gzip",
)
data_SMPS.columns

In [ ]:
data_HTRU_low_mid = pd.read_pickle(
    pathlib.Path().joinpath(output_dir, "survey_HTRU_low_mid_results.pkl.gz"),
    compression="gzip",
)
data_HTRU_low_mid.columns

Extracting the properties of the simulated neutron stars.

In [ ]:
r = data_full["r"]["[kpc]"].to_numpy()
phi = data_full["phi"]["[rad]"].to_numpy()
x = r * np.cos(phi)
y = r * np.sin(phi)
z = data_full["z"]["[kpc]"].to_numpy()
P = data_full["P"]["[s]"].to_numpy()
P_dot = data_full["P_dot"]["[s s^-1]"].to_numpy()
intercepted_radio = data_full["intercepted_radio"][" "].to_numpy(dtype=bool)

# Converting from galactocentric to Galactic coordinates.
l, b, _, _, _, _ = cc.galactocentric_to_galactic(
    x, y, z, np.zeros(len(x)), np.zeros(len(x)), np.zeros(len(x))
)

Extracting the indices of the pulsars detected in the individual surveys.

In [ ]:
idx_PMPS = data_PMPS["idx"].to_numpy(dtype=int)
idx_SMPS = data_SMPS["idx"].to_numpy(dtype=int)
idx_HTRU_low_mid = data_HTRU_low_mid["idx"].to_numpy(dtype=int)

## Plotting the simulation results

As a first diagnostic, we plot our population and corresponding detections in Galactic longitude and latitude.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    l,
    b,
    linestyle="None",
    marker="o",
    color="lightgray",
    markersize=2,
    alpha=1,
    rasterized=True,
    label=r"Simulation all",
)
ax.plot(
    l[intercepted_radio],
    b[intercepted_radio],
    linestyle="None",
    marker="o",
    color="gray",
    markersize=2,
    alpha=0.3,
    rasterized=True,
    label=r"Intercepting our LOS",
)

ax.plot(
    l[idx_PMPS],
    b[idx_PMPS],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=7,
    alpha=1,
    rasterized=True,
    label=r"Detected by PMPS",
)
ax.plot(
    l[idx_SMPS],
    b[idx_SMPS],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=5,
    alpha=1,
    rasterized=True,
    label=r"Detected by SMPS",
)
ax.plot(
    l[idx_HTRU_low_mid],
    b[idx_HTRU_low_mid],
    linestyle="None",
    marker="o",
    # fillstyle="none",
    color="tab:purple",
    markersize=5,
    alpha=1,
    rasterized=True,
    label=r"Detected by HTRU",
)

ax.plot(
    0.0,
    0.0,
    linestyle="None",
    marker="*",
    color="tab:orange",
    markersize=20,
    label="Galactic center",
)
ax.set_xlim(-180.0, 180.0)
ax.set_ylim(-90.0, 90.0)
ax.set_xlabel("l [deg]")
ax.set_ylabel("b [deg]")
plt.legend(
    bbox_to_anchor=(1, 1),
    frameon=False,
    loc=0,
    fontsize=20,
    markerscale=5,
    handler_map={
        PathCollection: HandlerPathCollection(update_func=update),
        plt.Line2D: HandlerLine2D(update_func=update),
    },
)
plt.show()

We can also visualize our population and corresponding detections in the $P-\dot{P}$ plane.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    P,
    P_dot,
    linestyle="None",
    marker="o",
    color="lightgray",
    markersize=2,
    alpha=1,
    rasterized=True,
    label="Simulation all",
)
ax.plot(
    P[intercepted_radio],
    P_dot[intercepted_radio],
    linestyle="None",
    marker="o",
    color="gray",
    markersize=2,
    alpha=0.3,
    rasterized=True,
    label="Intercepting our LOS",
)


ax.plot(
    P[idx_PMPS],
    P_dot[idx_PMPS],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=7,
    alpha=1.0,
    rasterized=True,
    label="Detected by PMPS",
)
ax.plot(
    P[idx_SMPS],
    P_dot[idx_SMPS],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=5,
    alpha=1.0,
    rasterized=True,
    label="Detected by SMPS",
)
ax.plot(
    P[idx_HTRU_low_mid],
    P_dot[idx_HTRU_low_mid],
    linestyle="None",
    marker="o",
    # fillstyle="none",
    color="tab:purple",
    markersize=5,
    alpha=1.0,
    rasterized=True,
    label="Detected by HTRU",
)

ax.set_xscale("log")
ax.set_yscale("log")

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$\dot{P}$ [s/s]")
plt.legend(
    bbox_to_anchor=(1, 1),
    frameon=False,
    loc=0,
    fontsize=20,
    markerscale=5,
    handler_map={
        PathCollection: HandlerPathCollection(update_func=update),
        plt.Line2D: HandlerLine2D(update_func=update),
    },
)

plt.show()

For more details explore the next tutorial notebook!